In [1]:
import numpy as np
from qewton.config.axes import BatchAxes, FeatureAxes, GeometryAxes
from qewton.visualization import *
from qewton import DataConfiguration
images = 255 * np.random.rand(20, 128, 128, 2)


from qewton.config import Variable
import qewton

batch_axes = BatchAxes(20)

batch = SliderSpec(
    batch_axes,  
    init_state=0,
    minimum=0,
    maximum=19,
    step=1,
)


X = Variable("x", dim=1)
Y = Variable("y", dim=1)
Z = Variable("z", dim=1)
C = Variable("c", dim=1)

data_config = DataConfiguration(
    batch_axes,
    GeometryAxes(qewton.geometries.Geometry(X*Y, shape=(128, 128))),
    FeatureAxes(Z * C))

plot = SurfacePlot(
    images,
    data_config,
    x = X,
    y = Y,
    z = AxisSpec(Z, log_scale=True),
    color = C,
    controls = [batch]
)

figure = Figure(plot, title="Image Plot")

app = DashApplication.create(figure)

app.run(debug=True, jupyter_mode="external")

Dash app running on http://127.0.0.1:8050/


In [6]:
from qewton.config.variables import Variable
from qewton.geometries.continuous.domains_2d.circle import Circle
from qewton.visualization.plots.geometry import GeometryPlot
from qewton.visualization.figure import Figure
from qewton.visualization import *

x = Variable("x", 2)  # 2D-Variable, wie von Box/Sphere/etc. verlangt

box = Circle(variable=x, center=[0, 0], radius=1.0)
plot = GeometryPlot(box, max_vertex_distance=0.2, show_edges=False)
fig = Figure(plot, title="Circle")


app = DashApplication.create(fig)
app.run(debug=True, jupyter_mode="external")

Dash app running on http://127.0.0.1:8050/


In [5]:
import numpy as np
import qewton
from qewton import DataConfiguration
from qewton.config import Variable
from qewton.config.axes import BatchAxes, FeatureAxes, GeometryAxes
from qewton.geometries.continuous.domains_2d.rectangle import Rectangle
from qewton.geometries.continuous.domains_3d.cylinder import Cylinder
from qewton.geometries.discrete.mesh_geometry import MeshGeometry
from qewton.visualization import *

x3 = Variable("x", 3)
U = Variable("u", 1)          # e.g. temperature, pressure, |velocity|

sphere = Cylinder(variable=x3, center=[0, 0, 0], radius=1.0, height=1.0)
mesh_geometry = sphere.create_mesh(max_vertex_distance=0.15)
vertices = mesh_geometry.mesh.vertices

# One scalar per vertex - here a smooth analytic field for testing
u = np.sin(3 * vertices[:, 0]) * np.cos(3 * vertices[:, 1])
data = u[:, None]                                    # shape (n_vertices, 1)

config = DataConfiguration(
    GeometryAxes(mesh_geometry),
    FeatureAxes(U),
)

plot = MeshFieldPlot(data, config, color=ColorSpec(U, cmap="viridis"), show_edges=False)
fig = Figure(plot, title="Field on sphere surface")

app = DashApplication.create(fig)
app.run(debug=True, jupyter_mode="external")

/tmp/ipykernel_578719/3808651055.py:19: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  u = np.sin(3 * vertices[:, 0]) * np.cos(3 * vertices[:, 1])


Dash app running on http://127.0.0.1:8050/


In [4]:
from qewton.geometries.continuous.domains_2d.circle import Circle
T = Variable("t", 1)
x2 = Variable("x", 2)
rect = Circle(variable=x2, center=[0, 0], radius=2.0)
mesh_geometry = rect.create_mesh(max_vertex_distance=0.05)
vertices = mesh_geometry.mesh.vertices

n_steps, n_vertices = 50, len(mesh_geometry.mesh.vertices)

# e.g. a decaying solution over time, shape (n_steps, n_vertices, 1)
solution = np.random.rand(n_steps, n_vertices, 1) * np.exp(
    -np.linspace(0, 3, n_steps)
)[:, None, None]

time_axes = BatchAxes(n_steps)
config = DataConfiguration(
    time_axes,
    GeometryAxes(mesh_geometry),
    FeatureAxes(T),
)

plot = MeshSurfacePlot(
    solution, config,
    z=T,
    controls=[SliderSpec(time_axes, 0, 0, 50)],   # min/max/init resolved from the config,
)

app = DashApplication.create(Figure(plot, title="Time evolution"))
app.run(debug=True, jupyter_mode="external")

Dash app running on http://127.0.0.1:8050/


In [1]:
import numpy as np
import qewton
from qewton import DataConfiguration
from qewton.config import Variable
from qewton.config.axes import GeometryAxes, FeatureAxes
from qewton.geometries.continuous.domains_2d.circle import Circle
from qewton.visualization import *

x2 = Variable('x', 2)
U = Variable('u', 1)
circle = Circle(variable=x2, center=[0, 0], radius=1.0)
mesh_geometry = circle.create_mesh(max_vertex_distance=0.3)
vertices = mesh_geometry.mesh.vertices
u1 = (np.sin(3 * vertices[:, 0]) * np.cos(3 * vertices[:, 1]))[:, None]
u2 = u1 * 5  # wider range

config = DataConfiguration(GeometryAxes(mesh_geometry), FeatureAxes(U))
shared = Scale()
plot_a = MeshFieldPlot(u1, config, color=ColorSpec(U, scale=shared), show_edges=False)
plot_b = MeshFieldPlot(u2, config, color=ColorSpec(U, scale=shared), show_edges=False)

fig = Figure([plot_a, plot_b], title='Shared mesh scale')

app = DashApplication.create(fig)
app.run(debug=True, jupyter_mode="external")

/tmp/ipykernel_650597/1692701058.py:14: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  u1 = (np.sin(3 * vertices[:, 0]) * np.cos(3 * vertices[:, 1]))[:, None]


Dash app running on http://127.0.0.1:8050/


In [4]:
import numpy as np

from qewton.visualization import *
from qewton.config.variables import Variable
from qewton.config.axes import GeometryAxes, FeatureAxes
from qewton import DataConfiguration
from qewton.geometries.continuous.domains_3d.cylinder import Cylinder
from qewton.geometries.discrete.plane_slice import plane_slice_geometry
from qewton.algorithms.building_blocks.geometry import MeshInterpolationNode
from qewton.visualization import EmbeddedGridPlot, ColorSpec, Scale, Figure, FixedSpec

# --- a real volumetric mesh: a cylinder, radius 1, height 2 ---
x3 = Variable("x", 3)
U = Variable("u", 1)  # the field we'll slice - e.g. temperature, pressure, |velocity|

cylinder = Cylinder(variable=x3, center=[0, 0, 0], radius=1.0, height=2.0)
mesh_geometry = cylinder.create_mesh(max_vertex_distance=0.15)

# an analytic field for testing: u = x^2 + y^2 + z^2 (radial distance squared)
vertices = np.asarray(mesh_geometry.mesh.vertices)
field = (vertices ** 2).sum(axis=1)
field = mesh_geometry.backend.build_tensor(field)  # match the mesh's backend/dtype

# --- build the slicing plane stack ---
# grid_variable is yours to name - the "height_level" child is what you'd later
# hand to a SliderSpec/FixedSpec to pick which slice is shown.
height_level = Variable("height_level", 1)
grid_variable = height_level * Variable("u_idx", 1) * Variable("v_idx", 1)

target_geometry = plane_slice_geometry(
    mesh_geometry,
    normal=[0, 0, 1],                    # horizontal slices
    offsets=[-0.7, 0.0, 0.7],            # three heights relative to the mesh centroid
    grid_variable=grid_variable,
    resolution=[40, 40] #-> derived automatically from the mesh's mean edge length
)

# --- interpolate the field onto the slice stack ---
node = MeshInterpolationNode(mesh_geometry, U, target_geometry, backend=mesh_geometry.backend)
sliced = node.forward(field)[..., None]  # (k, N1, N2, 1) - trailing feature axis for FeatureAxes(U)

# --- plot: three slices at once, at their true 3D positions, one shared color scale ---
config = DataConfiguration(GeometryAxes(target_geometry), FeatureAxes(U))
shared = Scale()
plots = [
    EmbeddedGridPlot(
        sliced, config, color=ColorSpec(U, cmap="viridis", scale=shared),
        controls=[FixedSpec(init_state=i, n_dimensions=1, variable_or_axes=height_level)],
    )
    for i in range(3)
]
fig = Figure(plots, title="Cylinder field, three horizontal slices")

app = DashApplication.create(fig)
app.run(debug=True, jupyter_mode="external")


Dash app running on http://127.0.0.1:8050/


In [2]:
import numpy as np

from qewton.config.variables import Variable
from qewton.config.axes import GeometryAxes, FeatureAxes, BatchAxes
from qewton import DataConfiguration
from qewton.geometries.continuous.domains_3d.cylinder import Cylinder
from qewton.geometries.discrete.plane_slice import PlaneSliceGeometry
from qewton.algorithms.building_blocks.geometry import MeshInterpolationNode
from qewton.visualization import (
    EmbeddedGridPlot, ColorSpec, Scale, Figure,
    FacetSpec, SliderSpec, DashApplication,
)
from qewton.backends.numpy.base import NumPyBackend

# --- a real volumetric mesh: a cylinder ---
x3 = Variable("x", 3)
cylinder = Cylinder(variable=x3, center=[0, 0, 0], radius=1.0, height=2.0, backend=NumPyBackend)
mesh_geometry = cylinder.create_mesh(max_vertex_distance=0.2)
print(mesh_geometry.mesh.vertices)
vertices = np.asarray(mesh_geometry.mesh.vertices)

# --- a time-varying field: a Gaussian blob that rises and fades ---
n_steps = 6
centers_z = np.linspace(-0.5, 1.5, n_steps)   # relative to the mesh centroid
amplitudes = np.linspace(1.0, 0.3, n_steps)

def field_at(step):
    dz = vertices[:, 2] - (vertices[:, 2].mean() + centers_z[step])
    r2 = vertices[:, 0] ** 2 + vertices[:, 1] ** 2 + dz ** 2
    return amplitudes[step] * np.exp(-3 * r2)

# --- three parallel slicing planes -> facet columns ---
height_level = Variable("height_level", 1)   # this is what the FacetSpec targets
grid_variable = height_level * Variable("u_idx", 1) * Variable("v_idx", 1)
target_geometry = PlaneSliceGeometry(
    mesh_geometry, normal=[0, 0, 1], offsets=[-0.5, 0.0, 0.5],
    grid_variable=grid_variable, resolution=[40,40], backend=mesh_geometry.backend,
)

U = Variable("u", 1)
node = MeshInterpolationNode(mesh_geometry, U, target_geometry, backend=mesh_geometry.backend)

# MeshInterpolationNode takes one field at a time, so loop over time steps
frames = [
    np.asarray(node.forward(mesh_geometry.backend.build_tensor(field_at(step))))
    for step in range(n_steps)
]
data = np.stack(frames)[..., None]   # (n_steps, k, N1, N2, 1)

# time lives OUTSIDE the geometry - it's not one of the grid's own dimensions
time_axes = BatchAxes(n_steps)
config = DataConfiguration(time_axes, GeometryAxes(target_geometry), FeatureAxes(U))

# --- facet over the 3 slice planes (columns), slider over time - on ONE plot ---
plot = EmbeddedGridPlot(
    data, config,
    color=ColorSpec(U, cmap="inferno", scale=Scale()),
    controls=[
        SliderSpec(time_axes, init_state=0, minimum=0, maximum=n_steps - 1),
        FacetSpec(height_level, orientation="col"),
    ],
)
fig = Figure(plot, title="Blob rising through three slice planes over time")

app = DashApplication.create(fig)
app.run(debug=True, jupyter_mode="external")


tensor([[ 1.0000e+00,  0.0000e+00,  0.0000e+00],
        [ 9.8079e-01,  1.9509e-01,  0.0000e+00],
        [ 9.2388e-01,  3.8268e-01,  0.0000e+00],
        ...,
        [-3.0386e-01,  3.7212e-17,  2.0000e+00],
        [-5.5819e-17, -3.0386e-01,  2.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  2.0000e+00]])
Dash app running on http://127.0.0.1:8050/


In [2]:
import numpy as np
from qewton import DataConfiguration
from qewton.config.variables import Variable
from qewton.config.axes import GeometryAxes, FeatureAxes, BatchAxes
from qewton.geometries.continuous.domains_3d.cylinder import Cylinder
from qewton.geometries.discrete.volume_grid import VolumeGridGeometry
from qewton.algorithms.building_blocks.geometry import MeshInterpolationNode
from qewton.visualization import QuiverPlot, VectorSpec, TimeSpec, Figure, DashApplication

# --- a real volumetric mesh: a cylinder, radius 1, height 2 ---
x3 = Variable("x", 3)
V3 = Variable("v", 3)  # the vector field, e.g. velocity

cylinder = Cylinder(variable=x3, center=[0, 0, 0], radius=1.0, height=2.0)
mesh_geometry = cylinder.create_mesh(max_vertex_distance=0.2)
vertices = np.asarray(mesh_geometry.mesh.vertices)

# --- the resampling grid is a structural constant, built once ---
i = Variable("i", 1); j = Variable("j", 1); k = Variable("k", 1)
grid_variable = i * j * k
target_geometry = VolumeGridGeometry(mesh_geometry, grid_variable, resolution=(14, 14, 14))
node = MeshInterpolationNode(mesh_geometry, V3, target_geometry, backend=mesh_geometry.backend)

# --- a time-varying swirl field: rotation phase modulated per step ---
n_steps = 10
per_step = []
for step in range(n_steps):
    phase = 2 * np.pi * step / n_steps
    swirl = np.stack([
        -vertices[:, 1] * np.cos(phase),
        vertices[:, 0] * np.cos(phase),
        np.sin(phase) * np.ones(len(vertices)),  # a vertical pulse component
    ], axis=1)
    field = mesh_geometry.backend.build_tensor(swirl)
    per_step.append(node.forward(field))  # interpolate this step's field onto the grid

data = mesh_geometry.backend.math.stack(per_step, axis=0)  # (n_steps, N1, N2, N3, 3)

# --- plot: an animated arrow field, scrubbed/played over time ---
step_axis = BatchAxes(n_steps)
config = DataConfiguration(step_axis, GeometryAxes(target_geometry), FeatureAxes(V3))

plot = QuiverPlot(
    data, config,
    vector=VectorSpec(V3, scale=0.2, color_by_magnitude=True),
    controls=[TimeSpec(step_axis, duration=400)],
)
fig = Figure(plot, title="Animated volume vector field")
app = DashApplication.create(fig)
app.run(debug=True, jupyter_mode="external")



Dash app running on http://127.0.0.1:8050/


In [1]:
from typing import Annotated
from qewton.graphs.graphs import Graph
from qewton.graphs.nodes import Node
from qewton.config.data_configurations import DataConfiguration
from qewton.config.axes import EllipsisAxes
from qewton.config.variables import Variable
from qewton.algorithms.dl_models.fcn import FCN
from qewton.algorithms.building_blocks.creation import Zeros
from qewton.backends import TensorType
from qewton.visualization import GraphPlot, Figure

X = Variable("x", 1)
U = Variable("u", 1)

# A minimal leaf node - LossNode just passes its input through, standing in
# for a real loss/constraint node for this demo.
class LossNode(Node):
    def forward(
        self, pred: Annotated[TensorType, DataConfiguration(EllipsisAxes())]
    ) -> Annotated[TensorType, DataConfiguration(EllipsisAxes())]:
        return pred

graph = Graph()
src = Zeros(shape=(1,), name="Source")
fcn = FCN(in_neurons=X, hidden_neurons=5, out_neurons=U, n_hidden_layers=2, name="fcn")
loss = LossNode(name="Loss")

graph.connect(src.output_ports[0], fcn.input_ports[0])
graph.connect(fcn.output_ports[0], loss.input_ports[0])
graph.sort()  # required - GraphPlot needs a sorted graph

# depth=0: fcn collapsed to a single box
Figure(GraphPlot(graph, depth=0, title="Collapsed")).show()

# depth=1: fcn's inner layers (Linear/ReLU chain) expanded inline
Figure(GraphPlot(graph, depth=1, title="Expanded FCN")).show()

# depth=2: one level deeper still - each Linear's own weight/bias/functional_linear
Figure(GraphPlot(graph, depth=2, title="Expanded FCN + Linear")).show()
